## 2. Imports and configuration

In [ ]:
from __future__ import annotations

import base64
import html
import io
import random
from pathlib import Path
from typing import Iterable

from PIL import Image, ImageEnhance, ImageFilter
from playwright.async_api import async_playwright

BASE_DIR = Path.cwd()

FONT_PATH = (
    BASE_DIR
    / "Kanz-al-Marjaan"
    / "build"
    / "Kanz-al-Marjaan-Regular.ttf"
)

PAIRS_PATH = BASE_DIR / "kanz_training_pairs.txt"

DATASET_DIR = BASE_DIR / "dataset"
IMAGE_DIR = DATASET_DIR / "images"

TRAIN_LABEL_PATH = DATASET_DIR / "train.txt"
VAL_LABEL_PATH = DATASET_DIR / "val.txt"
TEST_LABEL_PATH = DATASET_DIR / "test.txt"

CANVAS_WIDTH = 1800
CANVAS_HEIGHT = 230

VARIANTS_PER_LINE = 1 

TRAIN_RATIO = 0.85
VAL_RATIO = 0.10
TEST_RATIO = 0.05

RANDOM_SEED = 42

print("Base directory:", BASE_DIR)
print("Font path:", FONT_PATH)
print("Pairs path:", PAIRS_PATH)

In [ ]:
def check_required_files() -> None:
    if not FONT_PATH.is_file():
        raise FileNotFoundError(f"Font not found: {FONT_PATH}")

    if not PAIRS_PATH.is_file():
        raise FileNotFoundError(f"Training pairs file not found: {PAIRS_PATH}")

    print("Font found:", FONT_PATH)
    print("Training pairs found:", PAIRS_PATH)


check_required_files()

## 4. Load render-text and label pairs

In [ ]:
Pair = tuple[str, str]


def load_training_pairs(path: Path) -> list[Pair]:
    if not path.is_file():
        raise FileNotFoundError(f"Training pairs file not found: {path}")

    pairs: list[Pair] = []

    for line_number, raw_line in enumerate(
        path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if not raw_line.strip():
            continue

        parts = raw_line.split("\t", maxsplit=1)

        if len(parts) != 2:
            raise ValueError(
                f"Line {line_number} does not contain a tab separator: {raw_line!r}"
            )

        render_text = parts[0].strip()
        label_text = parts[1].strip()

        if not render_text or not label_text:
            continue

        pairs.append((render_text, label_text))

    if not pairs:
        raise ValueError("No valid training pairs were found.")

    return pairs


pairs = load_training_pairs(PAIRS_PATH)

print("Total pairs:", len(pairs))

for index, (render_text, label_text) in enumerate(pairs[:5], start=1):
    print(f"\nPair {index}")
    print("Render:", render_text)
    print("Label: ", label_text)

In [ ]:
def create_font_data_url(font_path: Path) -> str:
    if not font_path.is_file():
        raise FileNotFoundError(f"Font not found: {font_path}")

    font_bytes = font_path.read_bytes()

    if not font_bytes:
        raise ValueError("The font file is empty.")

    encoded = base64.b64encode(font_bytes).decode("ascii")
    return f"data:font/ttf;base64,{encoded}"


FONT_DATA_URL = create_font_data_url(FONT_PATH)

print("Data URL prefix:", FONT_DATA_URL[:30])
print("Data URL length:", len(FONT_DATA_URL))

## 6. Renderer class

In [ ]:
class KanzRenderer:
    def __init__(
        self,
        browser,
        font_data_url: str,
        canvas_width: int = 1800,
        canvas_height: int = 230,
    ) -> None:
        self.browser = browser
        self.font_data_url = font_data_url
        self.canvas_width = canvas_width
        self.canvas_height = canvas_height
        self.page = None

    async def start(self) -> None:
        self.page = await self.browser.new_page(
            viewport={
                "width": self.canvas_width,
                "height": self.canvas_height,
            },
            device_scale_factor=1,
        )

    async def close(self) -> None:
        if self.page is not None:
            await self.page.close()
            self.page = None

    async def render(
        self,
        text: str,
        font_size: int = 68,
        top_padding: int = 30,
        side_padding: int = 50,
    ) -> bytes:
        if self.page is None:
            raise RuntimeError("Call await renderer.start() before rendering.")

        safe_text = html.escape(text)

        document = f"""
        <!DOCTYPE html>
        <html lang="ur" dir="rtl">
        <head>
            <meta charset="UTF-8">

            <style>
                @font-face {{
                    font-family: "Kanz";
                    src: url("{self.font_data_url}") format("truetype");
                    font-weight: normal;
                    font-style: normal;
                    font-display: block;
                }}

                html, body {{
                    margin: 0;
                    padding: 0;
                    width: {self.canvas_width}px;
                    height: {self.canvas_height}px;
                    background: white;
                    overflow: hidden;
                }}

                #text-line {{
                    box-sizing: border-box;
                    width: 100%;
                    height: 100%;
                    padding: {top_padding}px {side_padding}px;

                    direction: rtl;
                    text-align: right;
                    white-space: nowrap;

                    font-family: "Kanz";
                    font-size: {font_size}px;
                    font-weight: normal;
                    line-height: 1.6;

                    color: black;
                    background: white;
                }}
            </style>
        </head>

        <body>
            <div id="text-line">{safe_text}</div>
        </body>
        </html>
        """

        await self.page.set_content(document, wait_until="load")

        await self.page.evaluate(
            f"""
            async () => {{
                await document.fonts.load('{font_size}px "Kanz"');
                await document.fonts.ready;
            }}
            """
        )

        font_loaded = await self.page.evaluate(
            f'document.fonts.check(\'{font_size}px "Kanz"\')'
        )

        if not font_loaded:
            raise RuntimeError("Chromium failed to load the Kanz font.")

        return await self.page.locator("#text-line").screenshot()

    async def render_randomized(self, text: str) -> bytes:
        return await self.render(
            text=text,
            font_size=random.randint(52, 76),
            top_padding=random.randint(24, 42),
            side_padding=random.randint(35, 65),
        )

## 7. Test one rendered image

This notebook uses the Playwright Async API. In Jupyter, call async functions with `await` directly.

In [ ]:
import subprocess
import sys
from pathlib import Path

from PIL import Image
from IPython.display import display


TEST_RENDER_PATH = BASE_DIR / "test_render.png"

test_render_text = pairs[0][0]
test_label_text = pairs[0][1]

command = [
    sys.executable,
    str(BASE_DIR / "render_worker.py"),
    str(FONT_PATH),
    str(TEST_RENDER_PATH),
    test_render_text,
    "70",
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
    encoding="utf-8",
)

print("Return code:", result.returncode)

if result.stdout:
    print("Output:")
    print(result.stdout)

if result.stderr:
    print("Errors:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Rendering subprocess failed."
    )

print("Saved:", TEST_RENDER_PATH)
print("Rendered source:", test_render_text)
print("Expected OCR label:", test_label_text)

display(Image.open(TEST_RENDER_PATH))

## 8. Image augmentation functions

In [ ]:
def load_grayscale_image(image_bytes: bytes) -> Image.Image:
    return Image.open(io.BytesIO(image_bytes)).convert("L")


def apply_rotation(
    image: Image.Image,
    maximum_angle: float = 1.0,
) -> Image.Image:
    angle = random.uniform(-maximum_angle, maximum_angle)

    return image.rotate(
        angle,
        expand=False,
        fillcolor=255,
    )


def apply_contrast(
    image: Image.Image,
    minimum: float = 0.82,
    maximum: float = 1.18,
) -> Image.Image:
    value = random.uniform(minimum, maximum)
    return ImageEnhance.Contrast(image).enhance(value)


def apply_blur(
    image: Image.Image,
    probability: float = 0.45,
) -> Image.Image:
    if random.random() >= probability:
        return image

    radius = random.uniform(0.15, 0.55)
    return image.filter(ImageFilter.GaussianBlur(radius))


def create_augmented_image(
    image_bytes: bytes,
    keep_clean: bool = False,
) -> Image.Image:
    image = load_grayscale_image(image_bytes)

    if keep_clean:
        return image

    image = apply_rotation(image)
    image = apply_contrast(image)
    image = apply_blur(image)

    return image

## 9. Test augmentation

In [ ]:
source_bytes = TEST_RENDER_PATH.read_bytes()

clean_image = create_augmented_image(
    source_bytes,
    keep_clean=True,
)

augmented_image = create_augmented_image(
    source_bytes,
    keep_clean=False,
)

clean_image.save(BASE_DIR / "test_clean.png")
augmented_image.save(BASE_DIR / "test_augmented.png")

print("Clean image")
display(clean_image)

print("Augmented image")
display(augmented_image)

## 10. Split dataset into train, validation, and test sets

In [ ]:
def split_dataset(
    pairs: list[Pair],
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
    random_seed: int = 42,
) -> tuple[list[Pair], list[Pair], list[Pair]]:
    total_ratio = train_ratio + val_ratio + test_ratio

    if abs(total_ratio - 1.0) > 0.00001:
        raise ValueError("Dataset ratios must add up to 1.0.")

    shuffled = pairs.copy()

    random_generator = random.Random(random_seed)
    random_generator.shuffle(shuffled)

    total = len(shuffled)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    return (
        shuffled[:train_end],
        shuffled[train_end:val_end],
        shuffled[val_end:],
    )


train_pairs, val_pairs, test_pairs = split_dataset(
    pairs=pairs,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    random_seed=RANDOM_SEED,
)

print("Total:", len(pairs))
print("Train:", len(train_pairs))
print("Validation:", len(val_pairs))
print("Test:", len(test_pairs))

assert len(train_pairs) + len(val_pairs) + len(test_pairs) == len(pairs)

## 11. PaddleOCR label helpers

In [ ]:
def create_label_entry(
    image_path: Path,
    label_text: str,
    base_directory: Path,
) -> str:
    relative_path = image_path.relative_to(base_directory).as_posix()
    return f"{relative_path}\t{label_text}"


def write_label_file(
    output_path: Path,
    entries: Iterable[str],
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_path.write_text(
        "\n".join(entries),
        encoding="utf-8",
    )

## 12. Generate one dataset split

In [ ]:
async def generate_split(
    renderer: KanzRenderer,
    split_name: str,
    pairs: list[Pair],
    image_directory: Path,
    base_directory: Path,
    variants_per_line: int,
) -> list[str]:
    image_directory.mkdir(parents=True, exist_ok=True)

    label_entries: list[str] = []

    for pair_index, (render_text, label_text) in enumerate(
        pairs,
        start=1,
    ):
        for variant_index in range(variants_per_line):
            screenshot_bytes = await renderer.render_randomized(
                render_text
            )

            image = create_augmented_image(
                screenshot_bytes,
                keep_clean=(variant_index == 0),
            )

            filename = (
                f"{split_name}_"
                f"{pair_index:06d}_"
                f"v{variant_index + 1:02d}.png"
            )

            image_path = image_directory / filename
            image.save(image_path)

            label_entries.append(
                create_label_entry(
                    image_path=image_path,
                    label_text=label_text,
                    base_directory=base_directory,
                )
            )

        if pair_index % 100 == 0:
            print(
                f"{split_name}: "
                f"{pair_index}/{len(pairs)} source lines processed"
            )

    return label_entries

## 13. Small-batch test

Generate only five source lines with two variants each. Inspect these before generating the complete dataset.

In [ ]:
import subprocess
import sys
from pathlib import Path

from PIL import Image
from IPython.display import display


SMALL_TEST_DIR = BASE_DIR / "test_dataset"
SMALL_TEST_IMAGE_DIR = SMALL_TEST_DIR / "images"
SMALL_TEST_LABEL_PATH = SMALL_TEST_DIR / "labels.txt"

SMALL_TEST_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

small_pairs = pairs[:5]
small_entries = []

variants_per_line = 1

for pair_index, (render_text, label_text) in enumerate(
    small_pairs,
    start=1,
):
    for variant_index in range(variants_per_line):
        filename = (
            f"sample_"
            f"{pair_index:06d}_"
            f"v{variant_index + 1:02d}.png"
        )

        image_path = SMALL_TEST_IMAGE_DIR / filename

        command = [
            sys.executable,
            str(BASE_DIR / "render_worker.py"),
            str(FONT_PATH),
            str(image_path),
            render_text,
            "70",
        ]

        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            encoding="utf-8",
        )

        if result.returncode != 0:
            print("Render text:", render_text)
            print("Worker output:", result.stdout)
            print("Worker error:", result.stderr)

            raise RuntimeError(
                f"Rendering failed for pair {pair_index}, "
                f"variant {variant_index + 1}"
            )

        # Keep the first version clean.
        # Apply augmentation to later versions.
        if variant_index > 0:
            image_bytes = image_path.read_bytes()

            augmented_image = create_augmented_image(
                image_bytes,
                keep_clean=False,
            )

            augmented_image.save(image_path)

        relative_path = image_path.relative_to(
            BASE_DIR
        ).as_posix()

        small_entries.append(
            f"{relative_path}\t{label_text}"
        )

        print("Created:", filename)

write_label_file(
    SMALL_TEST_LABEL_PATH,
    small_entries,
)

print("\nGenerated images:", len(small_entries))
print("Label file:", SMALL_TEST_LABEL_PATH)

for image_path in sorted(
    SMALL_TEST_IMAGE_DIR.glob("*.png")
)[:4]:
    print("\n", image_path.name)
    display(Image.open(image_path))

## 14. Inspect the small-batch labels

Each line should contain:

```text
image_path<TAB>correct Unicode OCR label
```

In [ ]:
print(
    SMALL_TEST_LABEL_PATH.read_text(
        encoding="utf-8"
    )
)

## 15. Generate the complete dataset

Only run this after the single-image and small-batch tests are correct.

In [ ]:
import random
import subprocess
import sys
from pathlib import Path


def generate_split_with_worker(
    split_name: str,
    split_pairs: list[tuple[str, str]],
    image_directory: Path,
    base_directory: Path,
    variants_per_line: int,
) -> list[str]:
    image_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    label_entries: list[str] = []

    for pair_index, (render_text, label_text) in enumerate(
        split_pairs,
        start=1,
    ):
        for variant_index in range(variants_per_line):
            filename = (
                f"{split_name}_"
                f"{pair_index:06d}_"
                f"v{variant_index + 1:02d}.png"
            )

            image_path = image_directory / filename

            font_size = random.randint(54, 76)

            command = [
                sys.executable,
                str(BASE_DIR / "render_worker.py"),
                str(FONT_PATH),
                str(image_path),
                render_text,
                str(font_size),
            ]

            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
            )

            if result.returncode != 0:
                print("Rendering failed")
                print("Split:", split_name)
                print("Pair:", pair_index)
                print("Variant:", variant_index + 1)
                print("Text:", render_text)
                print("stdout:", result.stdout)
                print("stderr:", result.stderr)

                raise RuntimeError(
                    f"Failed to render {split_name} "
                    f"pair {pair_index}, "
                    f"variant {variant_index + 1}"
                )

            if not image_path.is_file():
                raise FileNotFoundError(
                    f"Image was not created: {image_path}"
                )

            if variant_index > 0:
                image_bytes = image_path.read_bytes()

                augmented_image = create_augmented_image(
                    image_bytes,
                    keep_clean=False,
                )

                augmented_image.save(image_path)

            relative_path = image_path.relative_to(
                base_directory
            ).as_posix()

            label_entries.append(
                f"{relative_path}\t{label_text}"
            )

        if pair_index % 25 == 0 or pair_index == len(split_pairs):
            print(
                f"{split_name}: "
                f"{pair_index}/{len(split_pairs)} completed"
            )

    return label_entries

In [ ]:
def generate_complete_dataset() -> None:
    random.seed(RANDOM_SEED)

    IMAGE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Starting dataset generation")
    print("Train source lines:", len(train_pairs))
    print("Validation source lines:", len(val_pairs))
    print("Test source lines:", len(test_pairs))
    print("Variants per line:", VARIANTS_PER_LINE)

    train_entries = generate_split_with_worker(
        split_name="train",
        split_pairs=train_pairs,
        image_directory=IMAGE_DIR,
        base_directory=BASE_DIR,
        variants_per_line=VARIANTS_PER_LINE,
    )

    val_entries = generate_split_with_worker(
        split_name="val",
        split_pairs=val_pairs,
        image_directory=IMAGE_DIR,
        base_directory=BASE_DIR,
        variants_per_line=VARIANTS_PER_LINE,
    )

    test_entries = generate_split_with_worker(
        split_name="test",
        split_pairs=test_pairs,
        image_directory=IMAGE_DIR,
        base_directory=BASE_DIR,
        variants_per_line=VARIANTS_PER_LINE,
    )

    write_label_file(
        TRAIN_LABEL_PATH,
        train_entries,
    )

    write_label_file(
        VAL_LABEL_PATH,
        val_entries,
    )

    write_label_file(
        TEST_LABEL_PATH,
        test_entries,
    )

    print("\nDataset generation complete")
    print("Training images:", len(train_entries))
    print("Validation images:", len(val_entries))
    print("Test images:", len(test_entries))

In [ ]:
generate_complete_dataset()

## 16. Validate generated label files

In [ ]:
def validate_label_file(
    label_path: Path,
    base_directory: Path,
) -> None:
    missing_images: list[str] = []
    invalid_lines: list[int] = []
    total = 0

    for line_number, raw_line in enumerate(
        label_path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if not raw_line.strip():
            continue

        total += 1
        parts = raw_line.split("\t", maxsplit=1)

        if len(parts) != 2:
            invalid_lines.append(line_number)
            continue

        relative_path, label = parts

        if not label.strip():
            invalid_lines.append(line_number)

        image_path = base_directory / relative_path

        if not image_path.is_file():
            missing_images.append(relative_path)

    print("File:", label_path)
    print("Entries:", total)
    print("Invalid lines:", len(invalid_lines))
    print("Missing images:", len(missing_images))

    if invalid_lines:
        print("First invalid lines:", invalid_lines[:10])

    if missing_images:
        print("First missing images:", missing_images[:10])


for label_path in [
    TRAIN_LABEL_PATH,
    VAL_LABEL_PATH,
    TEST_LABEL_PATH,
]:
    validate_label_file(
        label_path=label_path,
        base_directory=BASE_DIR,
    )
    print()

## Expected final structure

```text
dataset/
├── images/
│   ├── train_000001_v01.png
│   ├── train_000001_v02.png
│   ├── val_000001_v01.png
│   └── test_000001_v01.png
├── train.txt
├── val.txt
└── test.txt
```

The PNG is rendered from the legacy Kanz sequence, while the label contains the intended normalized Unicode text.